In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("当前设备：", device)

当前设备： cuda


In [6]:
from obspy import read
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import os

# 自动选择设备：GPU（如果可用）或 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备：{device}")

# 加载并处理单个 .mseed 文件
def load_mseed_features(file_path, frame_length=1024, hop_length=512):
    try:
        st = read(file_path)
        tr = st[0]
        data = tr.data.astype(np.float32)

        # 简单归一化
        data = (data - np.mean(data)) / (np.std(data) + 1e-6)

        # 分帧
        frames = []
        for i in range(0, len(data) - frame_length, hop_length):
            frame = data[i:i+frame_length]
            frames.append(frame)

        return np.array(frames)

    except Exception as e:
        print(f"读取失败：{file_path}，错误：{e}")
        return np.empty((0, frame_length), dtype=np.float32)

# 批量加载文件夹中的所有 .mseed 文件
def load_dataset_from_mseed(folder_path):
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"路径不存在：{folder_path}")

    all_frames = []
    for fname in os.listdir(folder_path):
        if fname.endswith(".mseed"):
            full_path = os.path.join(folder_path, fname)
            frames = load_mseed_features(full_path)
            if frames.size > 0:
                all_frames.append(frames)

    if not all_frames:
        raise ValueError("未加载到任何有效的 .mseed 数据")

    return np.vstack(all_frames)

# 设置数据集路径
folder_path = r"C:\Users\22140\OneDrive\Desktop\Dataset\DataAINOA"

# 加载数据并迁移到 GPU
data_array = load_dataset_from_mseed(folder_path)
data_tensor = torch.tensor(data_array, dtype=torch.float32).to(device)
print(f"数据加载完成，形状为：{data_tensor.shape}")

# 构建 DataLoader（可选）
batch_size = 64
dataset = TensorDataset(data_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 示例模型（你可以替换为自己的模型）
class SimpleModel(torch.nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc = torch.nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.fc(x)

# 初始化并迁移模型到 GPU
model = SimpleModel(input_dim=1024).to(device)

# 示例前向传播
for batch in dataloader:
    inputs = batch[0].to(device)
    outputs = model(inputs)
    print(f"输出形状：{outputs.shape}")
    break  # 只跑一批作为示例


当前使用设备：cuda
数据加载完成，形状为：torch.Size([2363, 1024])
输出形状：torch.Size([64, 1])


In [7]:
pip install obspy torch numpy scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.


In [8]:
from obspy import read
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import os

def load_mseed_features(file_path, frame_length=1024, hop_length=512):
    st = read(file_path)
    tr = st[0]  # 取第一个通道
    data = tr.data.astype(np.float32)

    # 简单归一化
    data = (data - np.mean(data)) / (np.std(data) + 1e-6)

    # 分帧
    frames = []
    for i in range(0, len(data) - frame_length, hop_length):
        frame = data[i:i+frame_length]
        frames.append(frame)
    
    print(f"✔ 处理文件: {os.path.basename(file_path)}，提取帧数: {len(frames)}")
    return np.array(frames)

def load_dataset_from_mseed(folder_path):
    all_frames = []
    print("📂 开始读取 .mseed 文件...\n")
    for fname in os.listdir(folder_path):
        if fname.endswith(".mseed"):
            full_path = os.path.join(folder_path, fname)
            frames = load_mseed_features(full_path)
            all_frames.append(frames)
    print("\n✅ 所有文件处理完成！")
    return np.vstack(all_frames)

# 示例：加载数据并转换为 PyTorch 张量
data_array = load_dataset_from_mseed(r"C:\Users\22140\OneDrive\Desktop\Dataset\DataAINOA")
data_tensor = torch.tensor(data_array, dtype=torch.float32)
print(f"\n🎉 转换成功！最终数据张量形状: {data_tensor.shape}")

📂 开始读取 .mseed 文件...

✔ 处理文件: st_anomaly_flow_final.mseed，提取帧数: 772
✔ 处理文件: st_anomaly_no_flow_final.mseed，提取帧数: 807
✔ 处理文件: st_no_anomaly_final.mseed，提取帧数: 784

✅ 所有文件处理完成！

🎉 转换成功！最终数据张量形状: torch.Size([2363, 1024])


In [9]:
# -------------------- 📦 导入依赖 --------------------
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import os
from obspy import read

# -------------------- 📥 数据加载函数 --------------------
def load_mseed_features(file_path, frame_length=50, hop_length=25):
    st = read(file_path)
    tr = st[0]
    data = tr.data.astype(np.float32)
    data = (data - np.mean(data)) / (np.std(data) + 1e-6)  # 归一化

    frames = []
    for i in range(0, len(data) - frame_length, hop_length):
        frame = data[i:i+frame_length]
        frames.append(frame)
    print(f"✔ 处理文件: {os.path.basename(file_path)}，帧数: {len(frames)}")
    return np.array(frames)

def load_dataset_from_mseed(folder_path):
    all_frames = []
    print("📂 开始读取 .mseed 文件...\n")
    for fname in os.listdir(folder_path):
        if fname.endswith(".mseed"):
            full_path = os.path.join(folder_path, fname)
            frames = load_mseed_features(full_path)
            all_frames.append(frames)
    print("\n✅ 所有文件处理完成！")
    return np.vstack(all_frames)

# -------------------- 🎛️ GAN 生成器定义 --------------------
class AcousticGenerator(nn.Module):
    def __init__(self, noise_dim=100, output_dim=50):
        super(AcousticGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()  # 假设数据已归一化到 [-1, 1]
        )

    def forward(self, z):
        return self.model(z)

# -------------------- ⚙️ 物理约束损失函数 --------------------
def wave_equation_loss(p, c=1.0):
    d2p_dt2 = p[:, 2:] - 2 * p[:, 1:-1] + p[:, :-2]
    laplacian = d2p_dt2  # 简化为时间维度上的拉普拉斯
    residual = d2p_dt2 - c**2 * laplacian
    return torch.mean(residual**2)

def energy_conservation_loss(p, rho=1.0):
    v = p[:, 1:] - p[:, :-1]  # 粒子速度近似
    energy = 0.5 * rho * v**2
    return torch.mean(energy)

def elastic_constraint_loss(p):
    strain = p[:, 1:] - p[:, :-1]
    stress = strain  # 简化为线性关系
    return torch.mean(stress**2)

# -------------------- 🧪 总损失函数 --------------------
def total_physics_loss(fake_batch, real_batch, λ_wave=0.1, λ_energy=0.05, λ_elastic=0.05):
    data_loss = nn.MSELoss()(fake_batch, real_batch)
    wave_loss = wave_equation_loss(fake_batch)
    energy_loss = energy_conservation_loss(fake_batch)
    elastic_loss = elastic_constraint_loss(fake_batch)
    return data_loss + λ_wave * wave_loss + λ_energy * energy_loss + λ_elastic * elastic_loss

# -------------------- 🚀 主训练流程 --------------------
# ✅ 替换为你的数据路径
data_array = load_dataset_from_mseed(r"C:/Users/22140/OneDrive/Desktop/Dataset/DataAINOA")
data_tensor = torch.tensor(data_array, dtype=torch.float32)
print(f"\n🎉 数据转换成功！张量形状: {data_tensor.shape}")

dataset = TensorDataset(data_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

# 初始化生成器
noise_dim = 100
generator = AcousticGenerator(noise_dim=noise_dim, output_dim=50)
optimizer = torch.optim.Adam(generator.parameters(), lr=0.0002)

# 训练生成器
for epoch in range(100):
    for batch in loader:
        real_batch = batch[0]
        noise = torch.randn(real_batch.size(0), noise_dim)
        fake_batch = generator(noise)

        loss = total_physics_loss(fake_batch, real_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/100 | Total Loss: {loss.item():.4f}")


📂 开始读取 .mseed 文件...

✔ 处理文件: st_anomaly_flow_final.mseed，帧数: 15839
✔ 处理文件: st_anomaly_no_flow_final.mseed，帧数: 16559
✔ 处理文件: st_no_anomaly_final.mseed，帧数: 16079

✅ 所有文件处理完成！

🎉 数据转换成功！张量形状: torch.Size([48477, 50])
Epoch 1/100 | Total Loss: 1.0519
Epoch 2/100 | Total Loss: 1.1251
Epoch 3/100 | Total Loss: 0.9619
Epoch 4/100 | Total Loss: 0.9383
Epoch 5/100 | Total Loss: 1.0575
Epoch 6/100 | Total Loss: 0.7934
Epoch 7/100 | Total Loss: 1.1253
Epoch 8/100 | Total Loss: 1.2605
Epoch 9/100 | Total Loss: 1.3993
Epoch 10/100 | Total Loss: 0.9699
Epoch 11/100 | Total Loss: 1.6694
Epoch 12/100 | Total Loss: 0.8511
Epoch 13/100 | Total Loss: 1.2421
Epoch 14/100 | Total Loss: 0.6851
Epoch 15/100 | Total Loss: 0.9853
Epoch 16/100 | Total Loss: 0.9760
Epoch 17/100 | Total Loss: 1.1844
Epoch 18/100 | Total Loss: 1.1640
Epoch 19/100 | Total Loss: 1.1663
Epoch 20/100 | Total Loss: 0.7863
Epoch 21/100 | Total Loss: 0.9807
Epoch 22/100 | Total Loss: 1.4924
Epoch 23/100 | Total Loss: 0.9794
Epoch 24/100 |

In [10]:
real_data = torch.randn(1000, 50)  # 真实数据
noise = torch.randn(1000, 100)
fake_data = generator(noise).detach()  # 生成器输出      

In [ ]:
import torch
print(torch.cuda.is_available())

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# 合并数据
combined = torch.cat([real_data, fake_data], dim=0).numpy()
labels = np.array([0]*len(real_data) + [1]*len(fake_data))  # 0=真实，1=生成

# t-SNE 映射
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, random_state=42)
mapped = tsne.fit_transform(combined)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(mapped[labels==0, 0], mapped[labels==0, 1], c='blue', label='Real', alpha=0.5)
plt.scatter(mapped[labels==1, 0], mapped[labels==1, 1], c='red', label='Generated', alpha=0.5)
plt.legend()
plt.title("t-SNE Mapping of Real vs Generated Acoustic Data")
plt.xlabel("t-SNE Dim 1")
plt.ylabel("t-SNE Dim 2")
plt.grid(True)
plt.show()
print()


In [ ]:
print(“121323”)